In [ ]:
import os, sys, subprocess, importlib, shutil
import numpy as np

# --- Project setup (works on Colab and locally) ---
_PROJECT_ROOT = None
for _candidate in [os.getcwd(), "/content/Distance-Estimator-NN", "/home/adsarver/Distance-Estimator-NN"]:
    if os.path.isfile(os.path.join(_candidate, "data.py")):
        _PROJECT_ROOT = _candidate
        break
if _PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find project root containing data.py")

os.chdir(_PROJECT_ROOT)
sys.path.insert(0, _PROJECT_ROOT)

# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # not on Colab

# Pull dataset from Google Drive if not present
DATA_DIR = os.path.join(_PROJECT_ROOT, "rgbd-scenes-v2", "imgs")
if not os.path.isdir(DATA_DIR):
    DRIVE_ZIP = "/content/drive/MyDrive/rgbd-scenes-v2_imgs.zip"
    LOCAL_ZIP = os.path.join(_PROJECT_ROOT, "rgbd-scenes-v2_imgs.zip")
    if not os.path.isfile(DRIVE_ZIP):
        raise FileNotFoundError(f"Dataset zip not found on Google Drive at {DRIVE_ZIP}")
    print("Copying dataset from Google Drive ...")
    shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
    print("Unzipping ...")
    subprocess.check_call(["unzip", "-q", LOCAL_ZIP, "-d", _PROJECT_ROOT])
    os.remove(LOCAL_ZIP)
    print("Dataset ready.")

# Force-reload local modules in case they were cached
for _mod in ["data", "model"]:
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

print(f"Working directory: {os.getcwd()}")

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm

from data import RGBDDataset, scene_collate_fn
from model import DistanceNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Hyperparameters ---
IMG_SIZE = 480
HIDDEN_SIZE = 64
LSTM_LAYERS = 1
MEMORY_LENGTH = 30
MEMORY_STRIDE = 1
LR = 3e-4
EPOCHS = 10
TBTT_LEN = 4          # truncation window: backprop every N frames
VAL_SPLIT = 0.2        # fraction of scenes held out for validation

# --- Scene train/val split ---
from glob import glob
from collections import defaultdict

all_files = sorted(glob(os.path.join(DATA_DIR, "*/*-color.png")))
_scenes_map = defaultdict(list)
for f in all_files:
    _scenes_map[os.path.basename(os.path.dirname(f))].append(f)

all_scene_names = sorted(_scenes_map.keys())
np.random.seed(42)
np.random.shuffle(all_scene_names)

n_val = max(1, int(len(all_scene_names) * VAL_SPLIT))
val_scene_names = all_scene_names[:n_val]
train_scene_names = all_scene_names[n_val:]

# --- DataLoaders (one scene per batch) ---
train_ds = RGBDDataset(DATA_DIR, scene_names=train_scene_names)
val_ds   = RGBDDataset(DATA_DIR, scene_names=val_scene_names, random_seed=123)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=scene_collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, collate_fn=scene_collate_fn, num_workers=0)

print(f"Scenes — train: {len(train_ds)}, val: {len(val_ds)}")
print(f"  Train scenes: {train_scene_names}")
print(f"  Val scenes:   {val_scene_names}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/Distance-Estimator-NN
Using device: cuda
Scenes — train: 12, val: 2
  Train scenes: ['scene_01', 'scene_13', 'scene_06', 'scene_09', 'scene_03', 'scene_02', 'scene_14', 'scene_05', 'scene_08', 'scene_11', 'scene_04', 'scene_07']
  Val scenes:   ['scene_10', 'scene_12']


In [7]:
# --- Model ---
model = DistanceNN(
    hidden_size=HIDDEN_SIZE,
    lstm_num_layers=LSTM_LAYERS,
    memory_length=MEMORY_LENGTH,
    memory_stride=MEMORY_STRIDE,
    img_size=IMG_SIZE,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = GradScaler("cuda")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Model params: 236,231,289


In [ ]:
def run_epoch(loader, model, optimizer, scaler, device, is_train=True):
    """
    TBTT epoch with mixed-precision, tqdm progress bars, and validation metrics.
    Returns (avg_loss,) for train or (avg_loss, metrics_dict) for val.
    """
    mode = "Train" if is_train else "Val"
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_frames = 0

    # Validation metric accumulators
    sum_mae = 0.0
    sum_sq_err = 0.0
    sum_delta1 = 0
    total_val_pixels = 0

    scene_bar = tqdm(loader, desc=f"{mode} scenes", leave=True, file=sys.stdout)
    for si, scene in enumerate(scene_bar):
        rgb_list    = scene["rgb"]
        depth_list  = scene["depth"]
        rgb_crops   = scene["rgb_crops"]
        depth_crops = scene["depth_crops"]
        bboxes      = scene["bboxes"]
        crop_dims   = scene["crop_dims"]
        T = len(rgb_list)

        if T < 2:
            print("Too few frames in scene, skipping.")
            continue

        model.reset_lstm()
        chunk_loss = torch.tensor(0.0, device=device)
        chunk_frames = 0

        frame_bar = tqdm(range(T), desc=f"  {scene['scene']}", leave=False, file=sys.stdout)

        for t in frame_bar:
            crop_h, crop_w = crop_dims[t]
            if crop_h < 2 or crop_w < 2:
                print("Too small crop, skipping.")
                continue

            img = F.interpolate(rgb_list[t].unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
                                mode="bilinear", align_corners=False).to(device)
            bbox_t = bboxes[t].unsqueeze(0).to(device)
            obj_img = rgb_crops[t].unsqueeze(0).to(device)
            gt_depth = depth_crops[t].squeeze(0).to(device)

            gt_max = gt_depth.max()
            gt_norm = gt_depth / gt_max if gt_max > 0 else gt_depth

            if is_train:
                with autocast("cuda"):
                    pred = model(img, bbox_t, crop_h, crop_w, obj_img=obj_img).squeeze(0)
                    loss = F.mse_loss(pred, gt_norm)
                chunk_loss = chunk_loss + loss
            else:
                with torch.no_grad(), autocast("cuda"):
                    pred = model(img, bbox_t, crop_h, crop_w, obj_img=obj_img).squeeze(0)
                    chunk_loss = chunk_loss + F.mse_loss(pred, gt_norm)

                    # --- Per-pixel validation metrics ---
                    valid = gt_norm > 0
                    if valid.any():
                        p = pred[valid].float()
                        g = gt_norm[valid].float()
                        n_px = p.numel()

                        sum_mae += (p - g).abs().sum().item()
                        sum_sq_err += ((p - g) ** 2).sum().item()
                        ratio = torch.max(p / g.clamp(min=1e-6), g / p.clamp(min=1e-6))
                        sum_delta1 += (ratio < 1.25).sum().item()
                        total_val_pixels += n_px

            # Free GPU memory for frame tensors
            del img, bbox_t, obj_img, gt_depth, gt_norm, pred
            chunk_frames += 1
            total_frames += 1

            # --- TBTT boundary ---
            if is_train and chunk_frames % TBTT_LEN == 0:
                avg_loss = chunk_loss / TBTT_LEN
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(avg_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

                total_loss += chunk_loss.item()
                chunk_loss = torch.tensor(0.0, device=device)

                model.ctx_head.hx = tuple(h.detach() for h in model.ctx_head.hx)
                model.shape_head.hx = tuple(h.detach() for h in model.shape_head.hx)
                model.obj_head.hx = tuple(h.detach() for h in model.obj_head.hx)
                model.ctx_head.buffer = model.ctx_head.buffer.detach()
                model.shape_head.buffer = model.shape_head.buffer.detach()
                model.obj_head.buffer = model.obj_head.buffer.detach()

            frame_bar.set_postfix(loss=f"{total_loss / max(total_frames, 1):.5f}")

        frame_bar.close()

        # Flush remaining partial chunk
        leftover = chunk_frames % TBTT_LEN
        if leftover > 0:
            if is_train and chunk_loss.requires_grad:
                avg_loss = chunk_loss / leftover
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(avg_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            total_loss += chunk_loss.item()

        del chunk_loss
        torch.cuda.empty_cache()

        scene_bar.set_postfix(avg_loss=f"{total_loss / max(total_frames, 1):.5f}")

    scene_bar.close()

    avg_loss = total_loss / max(total_frames, 1)

    if not is_train and total_val_pixels > 0:
        metrics = {
            "mae":    sum_mae / total_val_pixels,
            "rmse":   (sum_sq_err / total_val_pixels) ** 0.5,
            "delta1": sum_delta1 / total_val_pixels,  # % pixels with max(p/g, g/p) < 1.25
        }
        return avg_loss, metrics

    return avg_loss, None

: 

In [9]:
# --- Training Loop (TBTT, scene-level streaming via DataLoader) ---
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*50}\nEpoch {epoch}/{EPOCHS}\n{'='*50}")
    
    train_loss, _ = run_epoch(train_loader, model, optimizer, scaler, device, is_train=True)
    val_loss, val_metrics = run_epoch(val_loader, model, optimizer, scaler, device, is_train=False)

    tag = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        tag = " * saved"

    print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}{tag}")
    if val_metrics:
        print(f"  Val MAE: {val_metrics['mae']:.5f} | "
              f"RMSE: {val_metrics['rmse']:.5f} | "
              f"δ<1.25: {val_metrics['delta1']:.2%}")


Epoch 1/10


Train scenes:   0%|          | 0/12 [00:00<?, ?it/s]

  scene_09:   0%|          | 0/732 [00:00<?, ?it/s]

  scene_13:   0%|          | 0/462 [00:00<?, ?it/s]

  scene_02:   0%|          | 0/834 [00:00<?, ?it/s]

: 

: 